# 🧠 C-Caption Transformer: a small transformer that writes and explains C programs

This notebook trains a character-level, decoder-only transformer (about 11M parameters) from scratch. It learns three things:
1. **Caption → code:** give it `/* Caption: Reverse a string in place. */` and it writes the program, then explains it in English.
2. **Code → explanation:** give it a C program and it writes `/* Explanation: ... */` for it.
3. **Question → answer:** give it `/* Question: What does malloc do? */` and it answers (for the C topics in the dataset).

**A simple form of reasoning, two ways:**
- **Plan first:** every training program has a numbered `/* Plan: ... */` before its code, so the model learns to write
  its steps first and then code from them (the idea behind "chain of thought").
- **Check and retry:** the 💬 cell at the end writes up to `TRIES` programs and keeps the first one gcc accepts.
- **Learn from your feedback:** mark a program 👎 and paste the correct code; the 🎓 cell fine-tunes a copy of the model on it.

Neither makes the model truly understand C, but you can see both at work: the plan comes first, and more output compiles.

**How to run**

0. Upload **`c_caption_dataset.txt`** to Google Drive into the folder `MyDrive/c_caption_transformer/`.
   (If it isn't there, the 📚 cell asks you to upload it from your computer instead.)
1. *Runtime → Change runtime type → **T4 GPU** → Save*.
2. **First a quick test:** tick `QUICK_TEST` in the ⚙️ **Settings** cell, then *Runtime → **Run all***.
   A tiny model runs through every cell in about a minute, so any error shows up straight away.
3. **The real run:** untick `QUICK_TEST` and *Run all* again. Training (8,000 steps) should take roughly 40–50 minutes on a T4;
   the training cell prints the real time per step and the minutes left.

**Checkpoints** are saved to Google Drive in `MyDrive/c_caption_transformer/<RUN_NAME>/`:
- `ckpt_iter_500.pt`, `ckpt_iter_1000.pt`, … every `SAVE_EVERY` steps (model weights only, about 45 MB each)
- `latest.pt`: model, optimizer and loss history. If Colab disconnects, just **Run all** again and training resumes from here.
- `best.pt`: the weights with the lowest validation loss

**About the dataset:** `c_caption_dataset.txt` holds 10,000 *different* small C programs from 36 task families, from
"add two numbers" and "hello world" to functions with 3–4 parameters, results returned through pointers or in a struct,
sorting, linked lists and file I/O. Names, types, values, loop styles and wording are randomised, and captions are phrased
in many ways ("Add two numbers.", "Write a program to add 2 numbers.", "How do I ... in C?"). Every program starts with
`/* Caption: ... */`, then a `/* Plan: ... */`, the code, an accurate English `/* Explanation: ... */` and `// END`. The file also has 876 short
Question/Answer documents on 73 C topics (`USE_QA` switches them on or off).

The model sees `BLOCK_SIZE = 1024` characters at a time. A typical program is about 700 characters, so while it writes the
explanation it can still see the caption and the code.

In [ ]:
#@title ⚙️ Settings
# Every setting lives in this one cell. The #@param markers turn the lines into
# form fields in Colab, so you can change values without editing the code.

QUICK_TEST = False               #@param {type:"boolean"}
# QUICK_TEST: tick it to run the whole notebook with a tiny model in ~1 minute (to catch errors early).

RUN_NAME = "c_captions_v3"      #@param {type:"string"}
# RUN_NAME: the folder name for this run's checkpoints. A new name = a fresh model.

SAVE_TO_DRIVE = True             #@param {type:"boolean"}
# SAVE_TO_DRIVE: keep checkpoints in Google Drive, so they survive when Colab disconnects.

RESUME = True                    #@param {type:"boolean"}
# RESUME: if this run already has checkpoints, continue training from the last one.

# --- data
DATA_FILE = "/content/drive/MyDrive/c_caption_transformer/c_caption_dataset.txt"  #@param {type:"string"}
MAX_PROGRAMS = 0                 #@param {type:"integer"}
# MAX_PROGRAMS: 0 = use every program in the file
USE_QA = True                    #@param {type:"boolean"}
# USE_QA: also train on the Question/Answer documents about C concepts in the dataset file
USE_SHAKESPEARE = False          #@param {type:"boolean"}
# comma-separated paths of extra text files to add ONCE (duplicate paragraphs are removed),
# e.g. "/content/drive/MyDrive/tiny_c_documentation.txt"
EXTRA_TEXT_FILES = ""            #@param {type:"string"}

# --- training
MAX_ITERS = 8000                 #@param {type:"integer"}   # total training steps
SAVE_EVERY = 500                 #@param {type:"integer"}   # write ckpt_iter_<step>.pt this often
EVAL_EVERY = 250                 #@param {type:"integer"}   # measure train/validation loss this often
LEARNING_RATE = 1e-3             #@param {type:"number"}    # how big each weight update is (at its peak)
BATCH_SIZE = 16                  #@param {type:"integer"}   # text windows the model learns from per step

# --- model (changing these needs a new RUN_NAME, old checkpoints won't fit)
BLOCK_SIZE = 1024  # characters of context: how far back the model can "see"
                   # (1024 fits a whole typical program, so it can see the code while writing the explanation)
N_EMBD = 384       # size of the vector that represents each character position
N_HEAD = 6         # attention heads per layer (each head is 384 / 6 = 64 numbers wide)
N_LAYER = 6        # number of transformer blocks stacked on top of each other
DROPOUT = 0.2      # randomly switch off 20% of activations while training, so it can't just memorise
EVAL_ITERS = 100   # batches averaged when measuring the loss (more = smoother numbers)
SEED = 1337        # fixes the random choices, so a rerun gives the same data split

if QUICK_TEST:   # tiny model + data: runs every cell in about a minute to catch errors early
    RUN_NAME += "_quicktest"   # separate folder, so the test never touches your real checkpoints
    MAX_PROGRAMS, MAX_ITERS, SAVE_EVERY, EVAL_EVERY, EVAL_ITERS = 300, 60, 20, 20, 5
    BATCH_SIZE, BLOCK_SIZE, N_EMBD, N_HEAD, N_LAYER = 16, 64, 64, 4, 2
    print(f"QUICK_TEST is ON: tiny settings, checkpoints go to '{RUN_NAME}'. Untick it for the real run.")

In [ ]:
#@title 🔧 Setup: GPU, Google Drive, checkpoint folder
import math, os, random, shutil, subprocess, tempfile, time
import torch                          # PyTorch: tensors (arrays that can live on the GPU) + automatic gradients
import torch.nn as nn                 # building blocks for neural networks (layers)
from torch.nn import functional as F  # stateless functions: softmax, cross_entropy, attention, ...

# google.colab only exists inside Colab; this lets the notebook also run elsewhere.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# "cuda" = the NVIDIA GPU. Training on the CPU would take many hours instead of minutes.
device = "cuda" if torch.cuda.is_available() else "cpu"
if IN_COLAB and device == "cpu":
    raise RuntimeError("No GPU! Go to Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all.")
use_amp = device == "cuda"          # mixed precision (float16): much faster on a T4
torch.manual_seed(SEED)             # same random numbers on every run (weights, batches)
random.seed(SEED)
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("Device: cpu (fine for a quick test, far too slow for real training)")

# Checkpoints go to Google Drive (survives disconnects) or, without Drive, to the Colab disk (deleted with the session).
if SAVE_TO_DRIVE and IN_COLAB:
    drive.mount("/content/drive")   # Colab asks you to allow access the first time
    CKPT_DIR = f"/content/drive/MyDrive/c_caption_transformer/{RUN_NAME}"
else:
    CKPT_DIR = os.path.abspath(f"c_caption_transformer/{RUN_NAME}")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoints folder:", CKPT_DIR)

## 1. Data
`c_caption_dataset.txt` holds the captioned C programs, each ending with a `// END` line.

In [ ]:
#@title 📚 Load the C dataset file and build the training text
import urllib.request

END = "// END"   # every program in the file ends with this line
# No file in Drive? Then ask for it with an upload button instead.
if not os.path.exists(DATA_FILE):
    if not IN_COLAB:
        raise FileNotFoundError(DATA_FILE)
    from google.colab import files
    print(f"{DATA_FILE} not found. Please choose {os.path.basename(DATA_FILE)} to upload:")
    DATA_FILE = os.path.abspath(next(iter(files.upload())))

# Cut the file into separate programs at every "// END" (and put the marker back on each one).
raw = open(DATA_FILE, encoding="utf-8").read()
programs = [p.strip() + f"\n{END}\n" for p in raw.split(END) if p.strip()]
if not USE_QA:
    programs = [p for p in programs if not p.startswith("/* Question:")]
n_qa = sum(p.startswith("/* Question:") for p in programs)
print(f"Loaded {len(programs) - n_qa:,} programs and {n_qa:,} Q&A documents from {DATA_FILE}")
if MAX_PROGRAMS:
    programs = programs[:MAX_PROGRAMS]
random.Random(SEED).shuffle(programs)   # mix the task types; the fixed SEED makes the order repeatable
n_val = max(1, len(programs) // 10)     # 10% of programs are held back for validation
# split by PROGRAM, so validation loss measures programs the model has never seen
train_docs, val_docs = programs[n_val:], programs[:n_val]


def dedupe_paragraphs(text):
    # Keep each paragraph only once (stops a repetitive file from being memorised).
    seen, keep = set(), []
    for para in text.split("\n\n"):
        if para.strip() and para not in seen:
            seen.add(para)
            keep.append(para)
    return "\n\n".join(keep)


# Optional extra text files (training data only).
for path in [p.strip() for p in EXTRA_TEXT_FILES.split(",") if p.strip()]:
    extra = dedupe_paragraphs(open(path, encoding="utf-8").read())
    train_docs.append(extra + "\n")
    print(f"Added {path}: {len(extra):,} chars after removing duplicate paragraphs")

# Optional Tiny Shakespeare (90% for training, the last 10% for validation).
if USE_SHAKESPEARE:
    if not os.path.exists("input.txt"):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "input.txt")
    shakespeare = open("input.txt", encoding="utf-8").read()
    cut = int(len(shakespeare) * 0.9)
    train_docs.append(shakespeare[:cut])
    val_docs.append(shakespeare[cut:])
    print(f"Added Tiny Shakespeare: {len(shakespeare):,} chars")

# The model reads one long string, so join the programs together.
train_text = "\n".join(train_docs)
val_text = "\n".join(val_docs)
# Quick health check: lots of unique lines = varied data the model can't simply memorise.
lines = [l for l in (train_text + val_text).splitlines() if l.strip()]
print(f"{len(programs):,} documents | train {len(train_text):,} chars | val {len(val_text):,} chars")
print(f"unique lines: {len(set(lines)):,} of {len(lines):,} | unique captions: {len({p.splitlines()[0] for p in programs})}")
print("\n----- example training program -----\n")
print(train_docs[0])

In [ ]:
#@title ✅ Check that the training programs compile (gcc)
# Colab has the gcc C compiler installed, so we can ask it whether code is valid C.
CC = shutil.which("gcc") or shutil.which("cc")


def compiles(code):
    # Syntax + type check only. Returns (ok, first error line).
    if CC is None:
        return None, "no C compiler found"
    with tempfile.TemporaryDirectory() as tmp:                  # throw-away folder
        src = os.path.join(tmp, "prog.c")
        with open(src, "w", encoding="utf-8") as f:
            f.write(code.replace(END, ""))                      # "// END" is our marker, not part of the program
        # -fsyntax-only: check the code without building a program
        r = subprocess.run([CC, "-std=c11", "-fsyntax-only", src], capture_output=True, text=True)
    errors = [l for l in r.stderr.splitlines() if "error" in l]
    return r.returncode == 0, (errors[0] if errors else "")    # exit code 0 = no errors


# Check 60 random training programs (all of them would take a while).
code_docs = [p for p in programs if p.startswith("/* Caption:")]   # Q&A documents are not programs
sample = random.Random(0).sample(code_docs, min(60, len(code_docs)))
results = [compiles(p) for p in sample]
if CC is None:
    print("No C compiler available - skipping the check.")
else:
    ok = sum(r[0] for r in results)
    print(f"{ok}/{len(sample)} sampled training programs compile")
    for (good, err), p in zip(results, sample):
        if not good:
            print("FAILED:", p.splitlines()[0], "->", err)

## 2. Model and training

In [ ]:
#@title 🔤 Character tokenizer + batches
# The model works with numbers, not letters. Each distinct character gets an id:
#   stoi = "string to integer"  e.g. {'a': 52, ...}      itos = "integer to string" (the reverse)
chars = sorted(set(train_text + val_text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}


def encode(s):
    # text -> list of ids
    return [stoi[c] for c in s if c in stoi]   # unknown characters are skipped


def decode(ids):
    # list of ids -> text
    return "".join(itos[i] for i in ids)


# Resuming? Reuse the checkpoint's vocabulary so character ids keep their meaning.
# (A changed vocabulary is what caused the "CUDA error: device-side assert" before.)
latest_path = os.path.join(CKPT_DIR, "latest.pt")
if RESUME and os.path.exists(latest_path):
    saved = torch.load(latest_path, map_location="cpu", weights_only=False)
    missing = set(chars) - set(saved["stoi"])
    if missing:
        raise ValueError(f"The data has characters the checkpoint's vocabulary lacks: {sorted(missing)}. "
                         "Use a new RUN_NAME (or RESUME = False) to start a fresh model.")
    stoi, itos = saved["stoi"], saved["itos"]
    del saved                                  # free the memory
vocab_size = len(stoi)                         # how many different characters the model can output
print("vocab size:", vocab_size)

# The whole dataset as one long tensor of ids, stored on the GPU so batches are fast to cut out.
train_data = torch.tensor(encode(train_text), dtype=torch.long, device=device)
val_data = torch.tensor(encode(val_text), dtype=torch.long, device=device)
offsets = torch.arange(BLOCK_SIZE, device=device)   # 0, 1, 2, ..., BLOCK_SIZE-1


def get_batch(split):
    # Cut BATCH_SIZE random windows of BLOCK_SIZE characters out of the text.
    # x = the input window, y = the same window shifted one character to the right.
    # So at every position the model must predict the NEXT character: that is all it is trained to do.
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (BATCH_SIZE, 1), device=device)   # random start positions
    x = data[ix + offsets]          # (B, T) windows of text
    y = data[ix + offsets + 1]      # the same windows shifted by one character
    return x, y

In [ ]:
#@title 🏗️ Model: decoder-only transformer
# Shapes used in the comments:  B = batch size, T = characters in the window (time), C = N_EMBD (channels)

class CausalSelfAttention(nn.Module):
    # Multi-head self-attention (MHSA): every character looks back at the earlier characters
    # and pulls in information from the ones that matter for predicting what comes next.
    # All heads at once; scaled_dot_product_attention uses fast fused kernels on the GPU.
    def __init__(self):
        super().__init__()
        # One layer makes the query, key and value vectors for all heads together:
        #   query = "what am I looking for?"  key = "what do I contain?"  value = "what I pass on"
        self.qkv = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.proj = nn.Linear(N_EMBD, N_EMBD)   # mixes the heads' outputs back together
        self.resid_drop = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)                       # each (B, T, C)
        # Split C into N_HEAD heads of size C // N_HEAD: (B, T, C) -> (B, N_HEAD, T, head_size).
        # Each head can learn to attend to something different (matching braces, variable names, ...).
        q, k, v = (t.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2) for t in (q, k, v))
        # softmax(q·k / sqrt(head_size)) · v  for every head.
        # is_causal=True hides future characters: position t may only look at positions 0..t.
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=DROPOUT if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)            # glue the heads back together
        return self.resid_drop(self.proj(y))


class Block(nn.Module):
    # One transformer block: attention (characters talk to each other),
    # then a small feed-forward network (each position "thinks" on its own).
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(N_EMBD)   # LayerNorm keeps the numbers in a stable range
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(N_EMBD)
        self.mlp = nn.Sequential(nn.Linear(N_EMBD, 4 * N_EMBD), nn.GELU(),
                                 nn.Linear(4 * N_EMBD, N_EMBD), nn.Dropout(DROPOUT))

    def forward(self, x):
        # "x + ..." is a residual connection: each layer ADDS to what it received instead of
        # replacing it. That keeps information flowing and makes deep networks trainable.
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, N_EMBD)   # learned vector for each character ("what")
        self.pos_emb = nn.Embedding(BLOCK_SIZE, N_EMBD)   # learned vector for each position ("where")
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])   # N_LAYER blocks in a row
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.head = nn.Linear(N_EMBD, vocab_size, bias=False)   # vector -> one score per possible next character
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # Start with small random weights: the model knows nothing at step 0.
        # Everything it will "know" about C is learned later by adjusting these numbers during training.
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        # idx: (B, T) character ids  ->  logits: (B, T, vocab_size) scores for the next character
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        logits = self.head(self.ln_f(self.blocks(x)))
        # Cross-entropy loss: how surprised the model was by the real next character
        # (lower = better guesses). Training is all about pushing this number down.
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


model = TransformerLanguageModel(vocab_size).to(device)   # create the model and move its weights to the GPU
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

In [ ]:
#@title 🚀 Train (saves a checkpoint every SAVE_EVERY steps, resumes automatically)
# AdamW decides how to change each weight, based on its gradient (the direction that lowers the loss).
# weight_decay gently pulls weights toward zero, which helps against memorising.
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.99), weight_decay=0.1)
# float16 numbers are fast but tiny gradients can round to zero; the GradScaler scales them up to prevent that.
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
WARMUP = 100                    # steps to ramp the learning rate up from almost zero
MIN_LR = LEARNING_RATE / 10     # the learning rate at the very end of training


def lr_at(step):
    # Linear warm-up, then cosine decay down to MIN_LR.
    # (Big steps early to learn fast, small steps late to fine-tune.)
    if step < WARMUP:
        return LEARNING_RATE * (step + 1) / WARMUP
    progress = min(1.0, (step - WARMUP) / max(1, MAX_ITERS - WARMUP))
    return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * progress))


@torch.no_grad()   # no gradients needed when we only measure
def estimate_loss():
    # Average loss over EVAL_ITERS batches, for training and validation text.
    # Validation = programs the model never trains on: the honest test of generalisation.
    model.eval()   # evaluation mode: switches dropout off
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                _, loss = model(*get_batch(split))
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()  # back to training mode
    return out


def save(path, step, full):
    # A checkpoint stores the learned weights plus the vocabulary (needed to decode the output).
    ckpt = {"model": model.state_dict(), "iter": step, "stoi": stoi, "itos": itos, "best_val": best_val,
            "config": dict(BLOCK_SIZE=BLOCK_SIZE, N_EMBD=N_EMBD, N_HEAD=N_HEAD, N_LAYER=N_LAYER, vocab_size=vocab_size)}
    if full:   # what's needed to resume training exactly
        ckpt.update(optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), history=history)
    tmp = path + ".tmp"
    torch.save(ckpt, tmp)
    os.replace(tmp, path)   # never leaves a half-written checkpoint behind


# Start from step 0, or continue from latest.pt if this run was trained before.
start_iter, best_val, history = 0, float("inf"), []
if RESUME and os.path.exists(latest_path):
    ckpt = torch.load(latest_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    start_iter, best_val, history = ckpt["iter"], ckpt["best_val"], ckpt["history"]
    del ckpt
    print(f"Resumed from {latest_path} at step {start_iter} (best val loss {best_val:.4f})")

if start_iter >= MAX_ITERS:
    print(f"Already trained to step {start_iter}. Raise MAX_ITERS to train further.")
else:
    print(f"Training steps {start_iter} -> {MAX_ITERS}")
model.train()
t0 = time.time()
# HOW THE MODEL LEARNS FROM NOTHING
# At step 0 the weights are random: the model knows no C, no English, not even that "{" needs a "}".
# Its guess for the next character is close to random among ~90 characters (loss ≈ 4.5).
# Every step it guesses the next character at BATCH_SIZE x BLOCK_SIZE positions of real text from the dataset;
# the loss measures how wrong those guesses were, backward() works out how each of the ~11M weights
# contributed to the mistakes, and the optimizer nudges every weight slightly so the correct characters
# become a bit more likely next time. Thousands of these tiny nudges add up to learned C patterns.
for it in range(start_iter, MAX_ITERS):
    for group in optimizer.param_groups:
        group["lr"] = lr_at(it)                      # set this step's learning rate
    xb, yb = get_batch("train")                      # 1) a random batch of text windows
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        _, loss = model(xb, yb)                      # 2) forward pass: predict, measure the loss
    optimizer.zero_grad(set_to_none=True)            # clear the previous step's gradients
    scaler.scale(loss).backward()                    # 3) backward pass: gradients for all ~11M weights
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # cap huge gradients so one bad batch can't wreck training
    scaler.step(optimizer)                           # 4) update the weights a tiny bit
    scaler.update()

    step = it + 1                                    # number of steps finished so far
    if step % EVAL_EVERY == 0 or step == MAX_ITERS:
        losses = estimate_loss()
        history.append((step, losses["train"], losses["val"]))
        per_step = (time.time() - t0) / (step - start_iter)
        print(f"step {step:5d} | train {losses['train']:.4f} | val {losses['val']:.4f} | "
              f"lr {lr_at(it):.2e} | {per_step * 1000:.0f} ms/step | ~{(MAX_ITERS - step) * per_step / 60:.1f} min left")
        if losses["val"] < best_val:                 # best on unseen programs so far -> keep it
            best_val = losses["val"]
            save(os.path.join(CKPT_DIR, "best.pt"), step, full=False)
    if step % SAVE_EVERY == 0 or step == MAX_ITERS:  # every 500 steps: a numbered checkpoint + latest.pt
        save(os.path.join(CKPT_DIR, f"ckpt_iter_{step}.pt"), step, full=False)
        save(latest_path, step, full=True)
        print(f"  💾 saved ckpt_iter_{step}.pt")

print("Checkpoints:", sorted(f for f in os.listdir(CKPT_DIR) if f.endswith(".pt")))

In [ ]:
#@title 📉 Loss curve
import matplotlib.pyplot as plt

# Both lines should go down. If "train" keeps falling while "validation" rises,
# the model is memorising the training programs instead of learning general patterns.
if history:
    steps, tr, va = zip(*history)
    plt.figure(figsize=(7, 3.5))
    plt.plot(steps, tr, label="train")
    plt.plot(steps, va, label="validation (unseen programs)")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
    plt.title("A growing gap between the lines = memorising instead of generalising")
    plt.savefig(os.path.join(CKPT_DIR, "loss_curve.png"), dpi=150, bbox_inches="tight")   # also saved to Drive, for the README
    plt.show()

## 3. Generate and check

In [ ]:
#@title ✍️ Use the model: write code, explain code, answer questions
#@markdown Which checkpoint to use: `best.pt`, `latest.pt` or e.g. `ckpt_iter_1500.pt`
CHECKPOINT = "best.pt"   #@param {type:"string"}
TEMPERATURE = 0.7        #@param {type:"number"}
TOP_K = 20               #@param {type:"integer"}
# TEMPERATURE: below 1 = safer, more predictable text; above 1 = more adventurous (and more mistakes).
# TOP_K: only pick from the 20 most likely next characters (cuts off unlikely nonsense).

ckpt = torch.load(os.path.join(CKPT_DIR, CHECKPOINT), map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])   # put the trained weights into the model
model.eval()                           # evaluation mode: no dropout
print(f"Loaded {CHECKPOINT} (step {ckpt['iter']})")


@torch.no_grad()
def generate(prompt, max_new_chars=1500):
    # Continue the prompt; stop when the model writes the END marker.
    # The model writes ONE character at a time, each time re-reading everything so far.
    text = prompt
    idx = torch.tensor([encode(text)], dtype=torch.long, device=device)
    for _ in range(max_new_chars):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits, _ = model(idx[:, -BLOCK_SIZE:])          # only the last BLOCK_SIZE characters fit
        logits = logits[:, -1, :].float() / TEMPERATURE      # scores for the NEXT character only
        if TOP_K:
            kth = torch.topk(logits, min(TOP_K, logits.size(-1))).values[:, [-1]]
            logits[logits < kth] = -float("inf")             # remove everything outside the top k
        # softmax turns scores into probabilities; multinomial picks one at random by those odds
        nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)                   # append it and go again
        text += itos[nxt.item()]
        if text.endswith(END):
            break
    return text


# All three uses are the same trick: start the text the way the training documents start,
# and let the model continue it.
def write_program(caption):
    # caption -> C program + English explanation
    return generate(f"/* Caption: {caption} */\n")


def explain_code(code):
    # C code -> English explanation: give the model the code, then the start of an explanation comment
    code = code.split("/* Explanation:")[0].split(END)[0].rstrip()
    return generate(code + "\n/* Explanation:", max_new_chars=700)[len(code) + 1:]


def ask(question):
    # question -> short answer (learned from the Question/Answer documents, if USE_QA was on)
    return generate(f"/* Question: {question} */\n/* Answer:", max_new_chars=700)


# 1) Caption -> code + explanation
SEEN = ["Reverse a string in place.",                 # captions like these are in the training data
        "Write a program to add 2 numbers.",
        "Find the minimum and maximum of an array with one function."]
UNSEEN = ["Find the second largest number in an array.",  # never in the training data
          "Count the uppercase letters in a string.",
          "Print the sum of the even numbers from 1 to 100.",
          # "twist" captions: a familiar task with one change. Does the plan (and the code) notice it?
          "Reverse an array but skip the first element.",
          "Count the vowels in a sentence, but not the letter e.",
          "Print the numbers from 1 to n, but only the multiples of 3."]

generated = {}
for caption in SEEN + UNSEEN:
    generated[caption] = write_program(caption)
    print("=" * 70, "\n" + ("[unseen caption]\n" if caption in UNSEEN else "") + generated[caption] + "\n")

# 2) Code -> explanation (this little program is not in the training data)
MY_CODE = r'''#include <stdio.h>

int triple(int x) {
    return 3 * x;
}

int main(void) {
    int a = 4, b = 9;
    printf("%d %d\n", triple(a), triple(b));
    return 0;
}'''
print("=" * 70, "\n[explain this code]\n" + MY_CODE + "\n" + explain_code(MY_CODE) + "\n")

# 3) Question -> answer
if USE_QA:
    for q in ["What does malloc do?", "How can a function return two values?",
              "What is a union?"]:          # the last topic is NOT in the Q&A data: expect a guess
        print("=" * 70, "\n" + ask(q) + "\n")

In [ ]:
#@title 🔍 Is the generated code valid, and is it new?
# Two honest checks for every generated program:
#   1) does gcc accept it as valid C?
#   2) did the model copy a training program, or write something new?
train_programs = set(programs)
train_lines = {l.strip() for p in programs for l in p.splitlines() if l.strip()}

print(f"{'caption':52} {'compiles':>9} {'exact copy':>11} {'new lines':>10}")
for caption, code in generated.items():
    ok, err = compiles(code)
    # count code lines (not the caption, the explanation or very short lines like "}") never seen in training
    code_part = code.split("/* Explanation:")[0]
    head, _, tail = code_part.partition("/* Plan:")
    code_part = head + tail.partition("*/")[2]          # drop the plan comment
    body = [l.strip() for l in code_part.splitlines()[1:] if len(l.strip()) > 15]
    new = sum(l not in train_lines for l in body)
    print(f"{caption[:50]:52} {str(ok):>9} {str(code in train_programs):>11} {new:>4}/{len(body):<5}")
    if ok is False:
        print("    first error:", err)
print("\n'new lines' = code lines (longer than 15 chars) that never appear in the training data.")
print("A good model compiles AND is not an exact copy: it recombines what it learned.")
print("Also read each explanation: does it describe what the code really does?")

## Tips
- **Continue training:** raise `MAX_ITERS` (e.g. to 12000) and **Run all**. Training resumes from `latest.pt`.
- **Compare checkpoints:** set `CHECKPOINT = "ckpt_iter_1000.pt"` in the ✍️ cell and run it again to see how the model improved over time.
- **Explain your own code:** `print(explain_code(my_code))` after the ✍️ cell. **Ask a question:** `print(ask("What is a pointer?"))`.
  The answers are only reliable for the 73 topics in the dataset; for anything else the model guesses in the same confident style.
- **Test the "reasoning":** in the 💬 cell try a twist caption such as *Reverse an array but skip the first element.* Does the plan mention the twist, and does the code follow it? Compare `TRIES = 1` with `TRIES = 5` to see how often check-and-retry rescues a program.
- **Teach it:** after a wrong program, use 📝 to paste the correct code, then run 🎓 and ask the same caption again. Try a slightly reworded caption too: corrections carry over to close rewordings, not to new tasks.
- **Overfitting?** If the validation loss rises while the training loss keeps falling, use the earlier `best.pt` or train on a bigger dataset file.
- **Add your own examples:** append programs to the dataset file in the same format (a `/* Caption: ... */` line, the code, an `/* Explanation: ... */` comment, then `// END`). Use a new `RUN_NAME` if they contain characters the dataset didn't have before.
- **A bigger or different dataset:** `make_c_dataset.py` (runs on any computer with Python) writes a new dataset file, e.g. `python make_c_dataset.py 15000 c_caption_dataset_v4.txt`.
- **Shakespeare:** `USE_SHAKESPEARE = True` brings back the Shakespeare + C blend. For clean C code it is better off: a model this small has limited capacity, and Shakespeare uses it on a different language.
- **Changing model size, `BLOCK_SIZE` or data characters** needs a new `RUN_NAME`; old checkpoints only fit the settings they were trained with.

In [ ]:
#@title 💬 Try the model (run the ✍️ cell first)
MODE = "Write code"   #@param ["Write code", "Ask a question"]
PROMPT = "Divide two integers and get both the quotient and the remainder."   #@param {type:"string"}
TRIES = 5   #@param {type:"integer"}
# TRIES: write up to this many programs and keep the first one gcc accepts (1 = no retry)


def clean(text):
    # remove the comment markers and the END marker, keep plain sentences
    return " ".join(text.replace("/*", "").replace("*/", "").replace(END, "").split())


if MODE == "Write code":
    for attempt in range(1, TRIES + 1):       # check and retry: write, test with gcc, try again
        out = write_program(PROMPT)
        ok, err = compiles(out)
        if ok:
            break
    last = {"caption": PROMPT, "output": out}   # remembered for the 📝 feedback cell
    code, _, explanation = out.partition("/* Explanation:")
    head, _, tail = code.partition("/* Plan:")
    plan, _, body = tail.partition("*/")
    print("🧠 Plan:" + plan.rstrip() + "\n")     # the steps the model wrote BEFORE its code
    print(head.rstrip() + "\n\n" + body.strip())
    print("\n💡 Explanation:", clean(explanation))
    print(f"\n✅ Compiles (attempt {attempt} of {TRIES})" if ok else f"\n❌ None of the {TRIES} attempts compiled: {err}")
else:
    print("💡", clean(ask(PROMPT).partition("/* Answer:")[2]))

## 4. Teach it with your feedback
Run the 💬 cell, then mark its program in the 📝 cell: **👍 Correct**, or **👎 Wrong** and paste the correct code
(it is checked with gcc and saved to `feedback.jsonl` on Drive). Then tick `LEARN_NOW` in the 🎓 cell and run it: it
fine-tunes a **copy** of the model on your feedback, mixed with normal training data, and shows the answer before and
after. With their default settings both cells do nothing, so *Run all* never saves feedback or starts learning by accident.

Be realistic: each correction teaches the model *that* caption (and close rewordings). A few corrections won't teach it to
solve other new tasks; that needs many, which is why AI companies collect feedback at a large scale.

In [ ]:
#@title 📝 Give feedback on the last program (run the 💬 cell first)
VERDICT = "Choose one"   #@param ["Choose one", "👍 Correct", "👎 Wrong"]
# "Choose one" saves nothing, so "Run all" never stores feedback by accident.
EXPLANATION = ""      #@param {type:"string"}
# EXPLANATION (optional): one sentence that says what the correct program does.
# Wrong? Paste the correct program between the ''' marks below (or leave it empty).
CORRECT_CODE = r'''

'''
import datetime, json

FEEDBACK_FILE = os.path.join(os.path.dirname(CKPT_DIR), "feedback.jsonl")   # next to the run folders, on Drive


def load_feedback():
    # All feedback saved so far, one entry per line.
    if not os.path.exists(FEEDBACK_FILE):
        return []
    with open(FEEDBACK_FILE, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def feedback_docs(entries):
    # Turn feedback into training documents in the same format as the dataset.
    docs = []
    for e in entries:
        if e["correct_code"]:                    # 👎 + your program: caption, your code, your explanation
            expl = f"/* Explanation: {e['explanation']} */\n" if e["explanation"] else ""
            docs.append(f"/* Caption: {e['caption']} */\n{e['correct_code']}\n{expl}{END}\n")
        elif e["verdict"] == "correct":          # 👍: the model's own program was right, keep it as an example
            docs.append(e["generated"].strip() + "\n")
    return docs


if VERDICT == "Choose one":
    print("Nothing saved: choose 👍 Correct or 👎 Wrong above, then run this cell.")
elif "last" not in globals():
    print("Run the 💬 cell in 'Write code' mode first, so there is a program to give feedback on.")
else:
    code = CORRECT_CODE.strip("\n").replace("\t", "    ")    # tabs -> 4 spaces, like the dataset
    ok, err = compiles(code) if code else (True, "")
    if ok is False:                                         # (None = no compiler available: save anyway)
        print("❌ Not saved: your code does not compile:", err)
    else:
        entry = {"time": datetime.datetime.now().isoformat(timespec="seconds"), "caption": last["caption"],
                 "generated": last["output"], "verdict": "correct" if VERDICT.startswith("👍") else "wrong",
                 "correct_code": code, "explanation": EXPLANATION.strip()}
        with open(FEEDBACK_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        entries = load_feedback()
        print(f"✅ Saved: '{entry['caption']}' marked {VERDICT}" + (" with your correct code." if code else "."))
        print(f"   {FEEDBACK_FILE} now holds {sum(bool(e['correct_code']) for e in entries)} corrections and "
              f"{sum(e['verdict'] == 'correct' and not e['correct_code'] for e in entries)} 👍 approvals.")
        if entry["verdict"] == "wrong" and not code:
            print("   Tip: also paste the correct code, otherwise the model has nothing to learn from.")

In [ ]:
#@title 🎓 Learn from feedback (optional: fine-tunes a COPY of the model, about 2-3 minutes)
LEARN_NOW = False      #@param {type:"boolean"}
# LEARN_NOW: tick it, then run this cell. Unticked, the cell does nothing (so "Run all" never starts learning).
FEEDBACK_STEPS = 300   #@param {type:"integer"}
FEEDBACK_LR = 1e-4     #@param {type:"number"}
FEEDBACK_SHARE = 0.25  # part of every batch that comes from your feedback; the rest is normal training text,
                       # so the model learns your corrections without forgetting everything else
assert "feedback_docs" in globals(), "Run the 📝 cell once first (it loads your saved feedback)."

docs = feedback_docs(load_feedback())
if not LEARN_NOW:
    print(f"Not learning: tick LEARN_NOW and run this cell. ({len(docs)} feedback documents are ready.)")
elif not docs:
    print("No corrections or 👍 approvals yet: use the 📝 cell first.")
else:
    unknown = sorted(set("".join(docs)) - set(stoi))
    if unknown:
        print("Note: these characters are not in the model's vocabulary and will be skipped:", unknown)
    # Repeat the feedback until it is long enough to cut BLOCK_SIZE-long training windows out of it.
    fb_text = "\n".join(docs)
    while len(fb_text) < 2 * BLOCK_SIZE + 2:
        fb_text += "\n" + "\n".join(docs)
    fb_data = torch.tensor(encode(fb_text), dtype=torch.long, device=device)
    n_fb = max(1, round(BATCH_SIZE * FEEDBACK_SHARE))

    def feedback_batch():
        x, y = get_batch("train")                                  # normal training windows ...
        ix = torch.randint(len(fb_data) - BLOCK_SIZE - 1, (n_fb, 1), device=device)
        x[:n_fb] = fb_data[ix + offsets]                           # ... with some rows replaced by feedback text
        y[:n_fb] = fb_data[ix + offsets + 1]
        return x, y

    # Start from the trained model; the original best.pt is never changed.
    best = torch.load(os.path.join(CKPT_DIR, "best.pt"), map_location=device, weights_only=False)
    model.load_state_dict(best["model"])
    model.eval()
    corrected = list(dict.fromkeys(e["caption"] for e in load_feedback() if e["correct_code"]))[-5:]
    before = {c: write_program(c) for c in corrected}             # answers BEFORE learning, to compare

    fb_optimizer = torch.optim.AdamW(model.parameters(), lr=FEEDBACK_LR, weight_decay=0.1)
    fb_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    model.train()
    for step in range(1, FEEDBACK_STEPS + 1):                     # the same training step as the 🚀 cell
        xb, yb = feedback_batch()
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            _, loss = model(xb, yb)
        fb_optimizer.zero_grad(set_to_none=True)
        fb_scaler.scale(loss).backward()
        fb_scaler.unscale_(fb_optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        fb_scaler.step(fb_optimizer)
        fb_scaler.update()
        if step % 50 == 0 or step == FEEDBACK_STEPS:
            print(f"feedback step {step}/{FEEDBACK_STEPS} | loss {loss.item():.4f}")
    model.eval()

    FB_DIR = CKPT_DIR + "_feedback"
    os.makedirs(FB_DIR, exist_ok=True)
    torch.save({"model": model.state_dict(), "iter": FEEDBACK_STEPS, "stoi": stoi, "itos": itos,
                "feedback_documents": len(docs)}, os.path.join(FB_DIR, "best.pt"))
    print(f"\n💾 Saved the improved copy to {FB_DIR}/best.pt (the original best.pt is unchanged).")
    print("   The 💬 cell now uses the improved model. To load it later, set CHECKPOINT in the ✍️ cell to "
          f"../{os.path.basename(FB_DIR)}/best.pt")

    for c in corrected:                                            # did it learn? same caption, before and after
        after = write_program(c)
        print("=" * 70, f"\n{c}\n--- before ---\n{before[c]}\n--- after ---\n{after}")
        print("compiles after:", compiles(after)[0])